In [ ]:
!pip install streamlit pyngrok pandas numpy

In [ ]:
%%writefile analytics_engine.py

import pandas as pd
import numpy as np

def find_col(df, possible_names):
    for name in possible_names:
        for col in df.columns:
            if name.lower() == col.lower():
                return col
    for name in possible_names:
        for col in df.columns:
            if name.lower() in col.lower():
                return col
    return None


# COLUMN FIX
# -----------------------------
def get_profit_col(df):
    if 'Order Profit Per Order' in df.columns:
        return 'Order Profit Per Order'
    elif 'Benefit per order' in df.columns:
        return 'Benefit per order'
    return find_col(df, ['profit'])


def get_discount_rate_col(df):
    if 'Order Item Discount Rate' in df.columns:
        return 'Order Item Discount Rate'
    return find_col(df, ['discount rate'])



# LOAD DATA
# -----------------------------
def load_data(file):
    try:
        df = pd.read_csv(file, encoding='utf-8')
    except:
        df = pd.read_csv(file, encoding='latin1')

    df.columns = (
        df.columns.str.strip()
        .str.replace('\xa0',' ',regex=True)
        .str.replace('\n',' ',regex=True)
        .str.replace('_',' ')
    )

    for col in df.columns:
        if any(x in col.lower() for x in ['sales','profit','discount']):
            df[col] = pd.to_numeric(df[col], errors='coerce')

    sales = find_col(df, ['Sales'])
    profit = get_profit_col(df)

    if sales and profit:
        df = df.dropna(subset=[sales, profit])
        df = df[df[sales] > 0]

    return df



# KPI
# -----------------------------
def compute_kpis(df):
    sales = find_col(df, ['Sales'])
    profit = get_profit_col(df)
    discount = find_col(df, ['Order Item Discount'])

    if sales is None or profit is None:
        return {"Revenue":0,"Profit":0,"Margin":0,"Discount Impact Ratio":0}

    revenue = df[sales].sum()
    prof = df[profit].sum()

    margin = (prof/revenue)*100 if revenue else 0

    discount_ratio = (
        df[discount].sum()/revenue*100
        if discount and revenue else 0
    )

    return {
        "Revenue": revenue,
        "Profit": prof,
        "Margin": margin,
        "Discount Impact Ratio": discount_ratio
    }



# TREND
# -----------------------------
def trend_analysis(df):
    date = find_col(df, ['Order Date','Date'])
    sales = find_col(df, ['Sales'])
    profit = get_profit_col(df)

    if None in [date,sales,profit]:
        return None

    df[date] = pd.to_datetime(df[date], errors='coerce')

    t = df.groupby(df[date].dt.to_period('M')).agg({
        sales:'sum', profit:'sum'
    }).reset_index()

    t[date] = t[date].astype(str)
    return t


# CUSTOMER
# -----------------------------
def customer_analysis(df):
    cust = find_col(df, ['Customer Id'])
    sales = find_col(df, ['Sales'])
    profit = get_profit_col(df)

    if None in [cust,sales,profit]:
        return pd.DataFrame()

    c = df.groupby(cust).agg({sales:'sum', profit:'sum'}).reset_index()

    c['Customer Value Index'] = c[profit]/c[sales]

    c['Score'] = (
        (c[sales]/c[sales].max()) +
        (c[profit]/c[profit].max())
    )

    c['Segment'] = pd.qcut(
        c['Score'], q=3,
        labels=['Low','Medium','High'],
        duplicates='drop'
    )

    return c.sort_values(by=profit, ascending=False)


def profit_concentration(cust):
    if cust.empty:
        return 0
    return (cust.iloc[:10,2].sum()/cust.iloc[:,2].sum())*100



# PRODUCT
# -----------------------------
def product_analysis(df):
    prod = find_col(df,['Product Name'])
    sales = find_col(df,['Sales'])
    profit = get_profit_col(df)

    if None in [prod,sales,profit]:
        return pd.DataFrame(),pd.DataFrame(),pd.DataFrame()

    p = df.groupby(prod).agg({sales:'sum',profit:'sum'}).reset_index()

    p['Margin (%)'] = (p[profit]/p[sales])*100

    loss = p[p['Margin (%)']<0]

    risky = p[
        (p[sales]>p[sales].quantile(0.75)) &
        (p['Margin (%)']<p['Margin (%)'].quantile(0.25))
    ]

    return p.sort_values(by='Margin (%)',ascending=False), loss, risky



# CATEGORY
# -----------------------------
def category_analysis(df):
    cat = find_col(df,['Category Name'])
    sales = find_col(df,['Sales'])
    profit = get_profit_col(df)
    market = find_col(df,['Market'])

    if None in [cat,sales,profit]:
        return pd.DataFrame(), None

    c = df.groupby(cat).agg({sales:'sum',profit:'sum'}).reset_index()
    c['Margin (%)'] = (c[profit]/c[sales])*100

    heat = None
    if market:
        temp = df.copy()
        temp['Margin'] = temp[profit]/temp[sales]
        heat = temp.pivot_table(values='Margin', index=cat, columns=market)

    return c, heat



# DISCOUNT
# -----------------------------
def discount_analysis(df, slider):
    df = df.copy()

    sales = find_col(df,['Sales'])
    profit = get_profit_col(df)
    disc = get_discount_rate_col(df)

    if None in [sales,profit]:
        return df

    df['Profit Ratio'] = df[profit]/df[sales]
    df['Simulated Profit'] = df[profit]*(1-slider)

    if disc:
        df['Discount Rate'] = df[disc]

    return df



# MARKET
# -----------------------------
def market_analysis(df):
    market = find_col(df,['Market'])
    sales = find_col(df,['Sales'])
    profit = get_profit_col(df)

    if None in [market,sales,profit]:
        return pd.DataFrame()

    m = df.groupby(market).agg({sales:'sum',profit:'sum'}).reset_index()
    m['Margin (%)'] = (m[profit]/m[sales])*100
    return m

# REGION
def region_analysis(df):
    region = find_col(df,['Order Region'])
    sales = find_col(df,['Sales'])
    profit = get_profit_col(df)

    if None in [region,sales,profit]:
        return pd.DataFrame()

    r = df.groupby(region).agg({sales:'sum',profit:'sum'}).reset_index()
    r['Margin (%)'] = (r[profit]/r[sales])*100
    return r

# COUNTRY
def country_analysis(df):
    country = find_col(df,['Order Country'])
    sales = find_col(df,['Sales'])
    profit = get_profit_col(df)

    if None in [country,sales,profit]:
        return pd.DataFrame()

    c = df.groupby(country).agg({sales:'sum',profit:'sum'}).reset_index()
    c['Margin (%)'] = (c[profit]/c[sales])*100
    return c



# SEGMENT
# -----------------------------
def segment_contribution(df):
    seg = find_col(df,['Customer Segment'])
    profit = get_profit_col(df)

    if None in [seg,profit]:
        return pd.Series()

    return df.groupby(seg)[profit].sum()

In [ ]:
%%writefile app.py
import streamlit as st
import matplotlib.pyplot as plt
import seaborn as sns
import os
import pandas as pd

from analytics_engine import *

st.set_page_config(layout="wide")
st.title("APL Logistics Profitability Dashboard")

DEFAULT_PATH = "APL_Logistics.csv"

df = None

if os.path.exists(DEFAULT_PATH):
    df = load_data(DEFAULT_PATH)
else:
    uploaded_file = st.sidebar.file_uploader("/content/APL_Logistics.csv", type=["csv"])
    if uploaded_file:
        df = load_data(uploaded_file)

if df is None:
    st.stop()

st.subheader("Data Preview")
st.dataframe(df.head())

if df.empty:
    st.error("Dataset empty")
    st.stop()


# FILTERS
# -----------------------------
def safe(col):
    return df[col].dropna().unique() if col in df.columns else []

segment = st.sidebar.multiselect("Segment", safe('Customer Segment'))
category = st.sidebar.multiselect("Category", safe('Category Name'))
market = st.sidebar.multiselect("Market", safe('Market'))
region = st.sidebar.multiselect("Region", safe('Order Region'))
product = st.sidebar.multiselect("Product", safe('Product Name'))

slider = st.sidebar.slider("Discount Simulation",0.0,1.0,0.2)

if segment: df = df[df['Customer Segment'].isin(segment)]
if category: df = df[df['Category Name'].isin(category)]
if market: df = df[df['Market'].isin(market)]
if region: df = df[df['Order Region'].isin(region)]
if product: df = df[df['Product Name'].isin(product)]

# KPI + EXECUTIVE SUMMARY
# -----------------------------
k = compute_kpis(df)

st.header("Executive Insights")

st.write(f"""
- Total Revenue: {k['Revenue']:.2f}
- Total Profit: {k['Profit']:.2f}
- Profit Margin: {k['Margin']:.2f}%
""")

gap = k['Revenue'] - k['Profit']
st.info(f"Revenue-Profit Gap: {gap:.2f}")

#Dynamic severity
if k['Margin'] < 5:
    st.error("Critical: Business operating at very low profitability")
elif k['Margin'] < 10:
    st.warning("Warning: Profit margins are under pressure")
else:
    st.success("Healthy profitability levels")

st.write("""
Key Observations:
- High revenue does not guarantee high profitability
- Discounts significantly impact margins
- Some customers and products drive disproportionate profit
""")

c1,c2,c3,c4 = st.columns(4)
c1.metric("Revenue",f"{k['Revenue']:.2f}")
c2.metric("Profit",f"{k['Profit']:.2f}")
c3.metric("Margin %",f"{k['Margin']:.2f}")
c4.metric("Discount Impact %",f"{k['Discount Impact Ratio']:.2f}")


# TREND
# -----------------------------
t = trend_analysis(df)
if t is not None:
    fig, ax = plt.subplots()
    ax.plot(t.iloc[:,0], t.iloc[:,1], label="Revenue")
    ax.plot(t.iloc[:,0], t.iloc[:,2], label="Profit")
    ax.legend()
    plt.xticks(rotation=45)
    st.pyplot(fig)

# CUSTOMER
# -----------------------------
cust = customer_analysis(df)
if not cust.empty:
    st.subheader("Customer Analysis")

    st.dataframe(cust.head(10))
    st.dataframe(cust.tail(10))

    #Profit concentration
    pc = profit_concentration(cust)
    st.info(f"Top 10 customers contribute {pc:.2f}% of total profit")

    loss_customers = cust[cust.iloc[:,2] < 0]
    if not loss_customers.empty:
        st.warning("Loss-making customers detected")
        st.dataframe(loss_customers)


# PRODUCT
# -----------------------------
prod, loss, risky = product_analysis(df)
if not prod.empty:
    st.subheader("Product Performance")

    st.dataframe(prod.head(10))

    if not loss.empty:
        st.warning("Loss-making products detected")
        st.dataframe(loss)

    if not risky.empty:
        st.warning("High revenue but low margin products detected")
        st.dataframe(risky)


# CATEGORY
# -----------------------------
cat, heat = category_analysis(df)

if not cat.empty:
    st.subheader("Category Performance")

    loss_cat = cat[cat['Margin (%)'] < 0]
    if not loss_cat.empty:
        st.warning("Loss-making categories detected")
        st.dataframe(loss_cat)

if heat is not None:
    fig, ax = plt.subplots()
    sns.heatmap(heat, ax=ax)
    st.pyplot(fig)


# MARKET
# -----------------------------
st.subheader("Market Performance")
m = market_analysis(df)
st.dataframe(m)

weak = m[
    (m.iloc[:,1] > m.iloc[:,1].quantile(0.75)) &
    (m['Margin (%)'] < 10)
]

if not weak.empty:
    st.warning("High revenue but low profit markets detected")
    st.dataframe(weak)

# DISCOUNT
# -----------------------------
disc = discount_analysis(df, slider)

if 'Discount Rate' in disc.columns:
    fig, ax = plt.subplots()
    ax.scatter(disc['Discount Rate'], disc['Profit Ratio'])
    st.pyplot(fig)

    avg_profit = df[get_profit_col(df)].mean()
    sim_profit = disc['Simulated Profit'].mean()

    st.info(f"Avg Profit Before Discount: {avg_profit:.2f}")
    st.info(f"Avg Profit After Simulation: {sim_profit:.2f}")

    threshold = disc[disc['Profit Ratio'] < 0]['Discount Rate'].mean()
    if pd.notna(threshold):
        st.warning(f"Discounts above {threshold:.2f} are likely causing losses")

    st.write("Higher discounts reduce margins and can lead to losses beyond threshold levels.")

# SEGMENT
# -----------------------------
seg = segment_contribution(df)
if not seg.empty:
    st.subheader("Customer Segment Contribution")
    st.bar_chart(seg)
    st.info("High-performing segments should be prioritized for growth")

# REGION / COUNTRY
# -----------------------------
st.subheader("Region Performance")
st.dataframe(region_analysis(df))

st.subheader("Country Performance")
st.dataframe(country_analysis(df))